# Object Detection

## Learning Objectives
1. Implement IoU and NMS from scratch in numpy
2. Build an anchor-based detector on synthetic bounding box data
3. Compute precision-recall curves and mAP from first principles
4. Understand FPN multi-scale detection and NMS parameter sensitivity


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: IoU and NMS from Scratch

In [ ]:
# ---- Level 1: IoU and NMS implemented from scratch in numpy ---------------
# IoU measures overlap between two boxes. NMS removes duplicate detections.
# These two functions underlie ALL object detection evaluation and inference.


def compute_iou(box1, box2):
    """Compute Intersection over Union (IoU) between two boxes.

    Args:
        box1: [x1, y1, x2, y2] (top-left and bottom-right corners)
        box2: [x1, y1, x2, y2]

    Returns:
        IoU scalar in [0, 1]. 0 = no overlap, 1 = perfect overlap.
    """
    # Intersection rectangle coordinates
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    # Area of intersection (zero if boxes do not overlap)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    # Area of each box
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    # Union = sum of areas minus overlap
    union_area = area1 + area2 - inter_area
    if union_area == 0:
        return 0.0
    return inter_area / union_area


def compute_iou_batch(boxes_a, boxes_b):
    """Vectorized IoU between every pair of boxes in boxes_a and boxes_b.

    Args:
        boxes_a: (N, 4) array of [x1, y1, x2, y2]
        boxes_b: (M, 4) array

    Returns:
        (N, M) IoU matrix
    """
    N = boxes_a.shape[0]
    M = boxes_b.shape[0]
    iou_matrix = np.zeros((N, M))
    for i in range(N):
        for j in range(M):
            iou_matrix[i, j] = compute_iou(boxes_a[i], boxes_b[j])
    return iou_matrix


def nms(boxes, scores, iou_threshold=0.5):
    """Non-Maximum Suppression: remove duplicate detections.

    Algorithm: greedily keep highest-scoring box; suppress all boxes
    that overlap it by more than iou_threshold.

    Args:
        boxes: (N, 4) array [x1, y1, x2, y2]
        scores: (N,) confidence scores
        iou_threshold: overlap threshold above which boxes are suppressed

    Returns:
        List of indices of surviving boxes
    """
    # Sort by score descending — always keep the most confident detection first
    order = np.argsort(-scores)
    kept = []
    suppressed = set()

    for idx in order:
        if idx in suppressed:
            continue
        kept.append(idx)
        # Suppress all remaining boxes that overlap this one
        for other in order:
            if other in suppressed or other == idx:
                continue
            if compute_iou(boxes[idx], boxes[other]) > iou_threshold:
                suppressed.add(other)
    return kept


# --- Demonstrate IoU on known boxes ------------------------------------
box_gt = np.array([10, 10, 50, 50])   # ground truth
box_perfect = np.array([10, 10, 50, 50])  # exact match
box_half = np.array([30, 10, 70, 50])     # half overlap
box_none = np.array([60, 60, 90, 90])     # no overlap

print('IoU Tests:')
print(f'  Perfect overlap:   {compute_iou(box_gt, box_perfect):.3f}  (expected 1.0)')
print(f'  Partial overlap:   {compute_iou(box_gt, box_half):.3f}  (expected ~0.25)')
print(f'  No overlap:        {compute_iou(box_gt, box_none):.3f}  (expected 0.0)')

# --- Demonstrate NMS ---------------------------------------------------
# Simulate 6 detections for one object — 4 duplicates, 1 separate object
test_boxes = np.array([
    [10, 10, 50, 50],   # best detection for object 1
    [11, 11, 51, 51],   # near-duplicate of object 1
    [13, 12, 53, 52],   # near-duplicate of object 1
    [12, 10, 52, 50],   # near-duplicate of object 1
    [70, 70, 110, 110], # separate object 2 (no overlap with obj 1)
    [72, 71, 112, 111], # near-duplicate of object 2
], dtype=float)
test_scores = np.array([0.95, 0.88, 0.82, 0.75, 0.90, 0.78])

kept_indices = nms(test_boxes, test_scores, iou_threshold=0.5)
print(f'NMS result: {len(test_boxes)} boxes → {len(kept_indices)} kept')
for i in kept_indices:
    print(f'  box {i}: {test_boxes[i]}, score={test_scores[i]:.2f}')

# Visualize detections before and after NMS
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = ['blue', 'cyan', 'teal', 'purple', 'red', 'orange']
for ax, title, idxs in [
    (ax1, 'Before NMS (all 6 detections)', range(len(test_boxes))),
    (ax2, f'After NMS ({len(kept_indices)} kept)', kept_indices),
]:
    ax.set_xlim(0, 130)
    ax.set_ylim(0, 130)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_aspect('equal')
    for i in idxs:
        b = test_boxes[i]
        rect = patches.Rectangle(
            (b[0], b[1]), b[2] - b[0], b[3] - b[1],
            linewidth=2, edgecolor=colors[i], facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(b[0], b[1] - 2, f'{test_scores[i]:.2f}',
                color=colors[i], fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/cv02_nms.png', dpi=80)
plt.close()
print('Saved: /tmp/cv02_nms.png')


## Level 2: Anchor-Based Detector on Synthetic Data

In [ ]:
# ---- Level 2: Simple anchor-based detector on synthetic rectangle data ----
# Generates 32x32 images each containing one rectangle of known position.
# The network predicts objectness + (cx, cy, w, h) offsets per anchor.


def generate_detection_data(n_samples=500, img_size=32):
    """Generate synthetic images with one rectangle per image.

    Each image has a random rectangle drawn by setting pixels to 1.0.
    Returns images (N, 1, H, W) and ground-truth boxes (N, 4) as [cx, cy, w, h]
    normalized to [0, 1].
    """
    images = np.zeros((n_samples, 1, img_size, img_size), dtype=np.float32)
    boxes = np.zeros((n_samples, 4), dtype=np.float32)  # [cx, cy, w, h] normalized
    for i in range(n_samples):
        # Random rectangle between 4 and 20 pixels wide/tall
        w = np.random.randint(4, img_size // 2)
        h = np.random.randint(4, img_size // 2)
        x1 = np.random.randint(0, img_size - w)
        y1 = np.random.randint(0, img_size - h)
        images[i, 0, y1:y1 + h, x1:x1 + w] = 1.0
        # Normalized center + size format used by all modern detectors
        boxes[i] = [
            (x1 + w / 2) / img_size,
            (y1 + h / 2) / img_size,
            w / img_size,
            h / img_size,
        ]
    return images, boxes


class SimpleDetector(nn.Module):
    """Minimal detector: CNN backbone + 2 heads (objectness + bbox).

    Objectness head: binary probability that an object is present.
    BBox head: predicted (cx, cy, w, h) normalized to [0, 1].
    """

    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                           # 16x16
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                           # 8x8
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                   # 1x1
        )
        # Objectness: probability that at least one object is in image
        self.obj_head = nn.Linear(64, 1)
        # Regression head: predict normalized (cx, cy, w, h) in [0,1]
        self.box_head = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 4), nn.Sigmoid()
        )

    def forward(self, x):
        features = self.backbone(x).flatten(1)   # (B, 64)
        objectness = self.obj_head(features)      # (B, 1) raw logit
        boxes = self.box_head(features)           # (B, 4) in [0,1]
        return objectness, boxes


# Generate data
X_det, y_det = generate_detection_data(n_samples=800)
X_tensor = torch.tensor(X_det)
y_tensor = torch.tensor(y_det)
# All images have exactly one object, so objectness is always 1
obj_tensor = torch.ones(len(X_det), 1)

det_dataset = TensorDataset(X_tensor, y_tensor, obj_tensor)
det_loader = DataLoader(det_dataset, batch_size=64, shuffle=True)

det_model = SimpleDetector().to(device)
det_opt = optim.Adam(det_model.parameters(), lr=1e-3)
bce_loss = nn.BCEWithLogitsLoss()  # for objectness
mse_loss = nn.MSELoss()             # for box regression (L2 / smooth-L1 in practice)

det_losses = []
for epoch in range(30):
    det_model.train()
    ep_loss = 0.0
    for X_b, box_b, obj_b in det_loader:
        X_b = X_b.to(device)
        box_b = box_b.to(device)
        obj_b = obj_b.to(device)
        det_opt.zero_grad()
        pred_obj, pred_box = det_model(X_b)
        # Combine objectness loss (classification) + box regression loss
        loss_obj = bce_loss(pred_obj, obj_b)
        loss_box = mse_loss(pred_box, box_b)
        # Lambda=5 weights box regression more heavily (common in YOLO)
        loss = loss_obj + 5.0 * loss_box
        loss.backward()
        det_opt.step()
        ep_loss += loss.item()
    det_losses.append(ep_loss / len(det_loader))

print('Detector training complete.')
# Evaluate box regression: compute mean IoU on validation slice
det_model.eval()
val_ious = []
with torch.no_grad():
    for i in range(50):
        img = X_tensor[i:i+1].to(device)
        gt_box = y_tensor[i].numpy()
        _, pred_box_t = det_model(img)
        pred = pred_box_t.cpu().numpy()[0]  # (cx,cy,w,h) normalized
        # Convert cx,cy,w,h to x1,y1,x2,y2 for IoU
        def cwh_to_xyxy(b, s=32):
            cx, cy, w, h = b * s
            return [cx - w/2, cy - h/2, cx + w/2, cy + h/2]
        iou = compute_iou(
            np.array(cwh_to_xyxy(gt_box)),
            np.array(cwh_to_xyxy(pred))
        )
        val_ious.append(iou)
print(f'Mean IoU on 50 validation images: {np.mean(val_ious):.3f}')
print(f'IoU > 0.5 rate: {np.mean(np.array(val_ious) > 0.5):.3f}')


## Real-World Example 1: NMS Threshold Sensitivity

In [ ]:
# ---- RW1: NMS IoU threshold sweep — shows trade-off between duplicates ----
# Too low: correct detections for nearby objects are merged (false negatives)
# Too high: duplicate detections remain (false positives)


def generate_dense_detections(n_objects=3, duplicates_per_obj=5, noise=3.0):
    """Simulate a detector's raw output for a scene with multiple objects.

    Each object has one correct detection plus several near-duplicate predictions
    with slightly different coordinates and lower confidence.
    """
    true_boxes = [
        [10, 10, 40, 40],   # object 1
        [50, 20, 80, 60],   # object 2
        [15, 70, 55, 110],  # object 3 — near object 1 vertically
    ]
    boxes = []
    scores = []
    for obj_id, tb in enumerate(true_boxes):
        for dup in range(duplicates_per_obj):
            # Jitter each duplicate by a small random offset
            jitter = np.random.uniform(-noise * (dup + 1), noise * (dup + 1), 4)
            b = np.array(tb, dtype=float) + jitter
            b = np.clip(b, 0, 130)
            boxes.append(b)
            # First duplicate has highest score; others decrease
            scores.append(0.95 - dup * 0.08 + np.random.uniform(-0.02, 0.02))
    return np.array(boxes), np.array(scores), true_boxes


boxes_raw, scores_raw, gt_boxes_list = generate_dense_detections()

thresholds = [0.2, 0.4, 0.5, 0.6, 0.8]
kept_counts = []
for thresh in thresholds:
    kept = nms(boxes_raw, scores_raw, iou_threshold=thresh)
    kept_counts.append(len(kept))
    print(f'NMS threshold={thresh:.1f}: {len(kept)} boxes kept'
          f' (true objects = 3)')

# Visualize at two extreme thresholds
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, thresh, title in zip(
    axes,
    [0.2, 0.5, 0.8],
    ['Low threshold (0.2)\nmay merge nearby objects',
     'Standard threshold (0.5)',
     'High threshold (0.8)\nkeeps duplicates'],
):
    kept = nms(boxes_raw, scores_raw, iou_threshold=thresh)
    ax.set_xlim(0, 135)
    ax.set_ylim(0, 135)
    ax.invert_yaxis()
    ax.set_title(f'{title}\n({len(kept)} boxes kept)')
    # Ground truth in green
    for gb in gt_boxes_list:
        rect = patches.Rectangle(
            (gb[0], gb[1]), gb[2]-gb[0], gb[3]-gb[1],
            linewidth=2, edgecolor='green', facecolor='none', linestyle='--'
        )
        ax.add_patch(rect)
    # Predictions in red
    for i in kept:
        b = boxes_raw[i]
        rect = patches.Rectangle(
            (b[0], b[1]), b[2]-b[0], b[3]-b[1],
            linewidth=1.5, edgecolor='tomato', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(b[0], b[1]-2, f'{scores_raw[i]:.2f}', color='tomato', fontsize=7)
plt.suptitle('NMS Threshold Sensitivity (green=GT, red=prediction)')
plt.tight_layout()
plt.savefig('/tmp/cv02_nms_threshold.png', dpi=80)
plt.close()
print('Saved: /tmp/cv02_nms_threshold.png')


## Real-World Example 2: FPN Multi-Scale Detection

In [ ]:
# ---- RW2: Feature Pyramid Network (FPN) — detect objects at 3 scales -----
# FPN merges semantically rich deep features with spatially precise shallow
# features via lateral connections and top-down upsampling.


class FPN(nn.Module):
    """Simplified FPN that produces feature maps at 3 scales (P3, P4, P5).

    Architecture:
    - Bottom-up: 3 stages that halve spatial resolution
    - Top-down: upsample deep features and add shallow features via 1x1 conv
    - Output: 3 feature pyramid levels, each with 128 channels
    """

    def __init__(self, in_ch=1, fpn_ch=128):
        super().__init__()
        # Bottom-up pathway: extract features at 3 scales
        self.stage3 = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
        )  # output: H/2 x W/2
        self.stage4 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(),
        )  # output: H/4 x W/4
        self.stage5 = nn.Sequential(
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.ReLU(),
        )  # output: H/8 x W/8
        # Lateral 1x1 convs to equalize channel counts across scales
        self.lat3 = nn.Conv2d(64, fpn_ch, 1)
        self.lat4 = nn.Conv2d(128, fpn_ch, 1)
        self.lat5 = nn.Conv2d(256, fpn_ch, 1)
        # Smoothing convs after top-down merging
        self.smooth3 = nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1)
        self.smooth4 = nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1)

    def forward(self, x):
        # Bottom-up: extract features at 3 spatial scales
        c3 = self.stage3(x)    # H/2 x W/2
        c4 = self.stage4(c3)   # H/4 x W/4
        c5 = self.stage5(c4)   # H/8 x W/8
        # Top-down: start from deepest, upsample and merge
        p5 = self.lat5(c5)     # deepest pyramid level
        # Upsample p5 to c4 resolution and add lateral connection
        p4 = self.lat4(c4) + nn.functional.interpolate(
            p5, size=c4.shape[2:], mode='nearest'
        )
        p4 = self.smooth4(p4)
        # Upsample p4 to c3 resolution and add lateral connection
        p3 = self.lat3(c3) + nn.functional.interpolate(
            p4, size=c3.shape[2:], mode='nearest'
        )
        p3 = self.smooth3(p3)
        return p3, p4, p5  # fine to coarse scales


# Verify FPN output shapes for a 64x64 input
fpn_model = FPN(in_ch=1, fpn_ch=128).to(device)
dummy_input = torch.randn(2, 1, 64, 64).to(device)  # batch=2, 1-channel, 64x64
with torch.no_grad():
    p3_out, p4_out, p5_out = fpn_model(dummy_input)

print('FPN output shapes:')
print(f'  P3 (fine, stride 2): {tuple(p3_out.shape)}')
print(f'  P4 (mid, stride 4):  {tuple(p4_out.shape)}')
print(f'  P5 (coarse, stride 8):{tuple(p5_out.shape)}')
print('Small objects detected at P3 (high resolution)')
print('Large objects detected at P5 (large receptive field)')

# Show conceptual FPN channel depth vs spatial size
fig, ax = plt.subplots(figsize=(8, 4))
levels = ['P3 (fine)\n32x32', 'P4 (mid)\n16x16', 'P5 (coarse)\n8x8']
channel_counts = [128, 128, 128]
spatial_sizes = [p3_out.shape[2]**2, p4_out.shape[2]**2, p5_out.shape[2]**2]
x_pos = range(3)
ax2 = ax.twinx()
bars1 = ax.bar(x_pos, channel_counts, width=0.4, align='center',
               color='steelblue', alpha=0.7, label='Channels')
bars2 = ax2.bar([p + 0.4 for p in x_pos], spatial_sizes, width=0.4, align='center',
                color='coral', alpha=0.7, label='Spatial elements')
ax.set_xticks([p + 0.2 for p in x_pos])
ax.set_xticklabels(levels)
ax.set_ylabel('Channels (blue)')
ax2.set_ylabel('Spatial elements HxW (red)')
ax.set_title('FPN: Equal channels at all scales, decreasing spatial resolution')
plt.tight_layout()
plt.savefig('/tmp/cv02_fpn.png', dpi=80)
plt.close()
print('Saved: /tmp/cv02_fpn.png')


## Real-World Example 3: mAP Computation from Scratch

## Comparison: Precision-Recall at Different IoU Thresholds

In [ ]:
# ---- RW3 + Comparison: mAP from scratch + PR at multiple IoU thresholds --
# mAP is the area under the precision-recall curve averaged over classes.
# Computed separately at IoU=0.5 and IoU=0.75 to see localization quality.


def compute_ap(precisions, recalls):
    """Compute Average Precision using the 11-point interpolation (VOC 2007).

    Interpolates precision at 11 recall levels: 0, 0.1, 0.2, ..., 1.0.
    AP = mean of max precision at each recall level.
    """
    ap = 0.0
    for recall_thresh in np.linspace(0, 1, 11):
        # Precision at this recall level = max precision at recall >= threshold
        precs_above = [
            p for p, r in zip(precisions, recalls) if r >= recall_thresh
        ]
        ap += max(precs_above) if precs_above else 0.0
    return ap / 11.0


def simulate_detector_output(n_gt=50, n_preds=200, iou_noise=0.15):
    """Simulate detector predictions vs ground truth for one class.

    Returns (scores, is_tp) where is_tp[i] = True if prediction i
    matches a ground truth box at IoU >= threshold.
    """
    # Simulate: 40% of predictions match GT, rest are false positives
    n_tp = int(n_preds * 0.40)
    n_fp = n_preds - n_tp
    # TP predictions have high base confidence + small noise
    tp_scores = np.clip(np.random.beta(5, 2, n_tp), 0.1, 1.0)
    fp_scores = np.clip(np.random.beta(2, 5, n_fp), 0.0, 0.9)
    scores = np.concatenate([tp_scores, fp_scores])
    is_tp = np.concatenate([np.ones(n_tp, dtype=bool),
                            np.zeros(n_fp, dtype=bool)])
    # Shuffle so they're not sorted by TP/FP status
    order = np.argsort(-scores)
    return scores[order], is_tp[order], n_gt


def precision_recall_curve(scores, is_tp, n_gt):
    """Build precision-recall curve from sorted detector outputs.

    Args:
        scores: confidence scores sorted descending
        is_tp: boolean array indicating true positives
        n_gt: total number of ground truth boxes

    Returns:
        (precisions, recalls) arrays
    """
    tp_cumsum = np.cumsum(is_tp)
    fp_cumsum = np.cumsum(~is_tp)
    n_preds = np.arange(1, len(scores) + 1)
    precisions = tp_cumsum / n_preds
    # Recall = TPs found so far / total GT objects
    recalls = tp_cumsum / n_gt
    return precisions, recalls


# Simulate output for 3 classes with different difficulty
class_configs = [
    ('Class A (easy)', 0.6, 'steelblue'),
    ('Class B (medium)', 0.35, 'orange'),
    ('Class C (hard)', 0.15, 'tomato'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
aps_050, aps_075 = [], []

for cls_name, tp_rate, color in class_configs:
    n_tp = int(200 * tp_rate)
    n_fp = 200 - n_tp
    tp_scores = np.clip(np.random.beta(5, 2, n_tp), 0.1, 1.0)
    fp_scores = np.clip(np.random.beta(2, 5, n_fp), 0.0, 0.9)
    scores = np.concatenate([tp_scores, fp_scores])
    is_tp = np.concatenate([np.ones(n_tp, bool), np.zeros(n_fp, bool)])
    order = np.argsort(-scores)
    scores_s, is_tp_s = scores[order], is_tp[order]
    prec, rec = precision_recall_curve(scores_s, is_tp_s, n_gt=50)
    ap050 = compute_ap(prec, rec)
    aps_050.append(ap050)
    # Simulate stricter IoU=0.75: fewer TPs survive
    n_tp_strict = int(n_tp * 0.55)
    n_fp_strict = 200 - n_tp_strict
    tp_s2 = np.clip(np.random.beta(4, 2, n_tp_strict), 0.1, 1.0)
    fp_s2 = np.clip(np.random.beta(2, 6, n_fp_strict), 0.0, 0.9)
    sc2 = np.concatenate([tp_s2, fp_s2])
    tp2 = np.concatenate([np.ones(n_tp_strict, bool), np.zeros(n_fp_strict, bool)])
    ord2 = np.argsort(-sc2)
    prec2, rec2 = precision_recall_curve(sc2[ord2], tp2[ord2], n_gt=50)
    ap075 = compute_ap(prec2, rec2)
    aps_075.append(ap075)
    axes[0].plot(rec, prec, label=f'{cls_name} (AP={ap050:.2f})', color=color)
    axes[1].plot(rec2, prec2, label=f'{cls_name} (AP={ap075:.2f})', color=color,
                 linestyle='--')

for ax, title in zip(axes, ['mAP@IoU=0.5', 'mAP@IoU=0.75']):
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)

print(f'mAP@0.50 = {np.mean(aps_050):.3f}')
print(f'mAP@0.75 = {np.mean(aps_075):.3f}')
print('mAP@0.75 is always <= mAP@0.50: stricter IoU threshold accepts fewer TPs.')
plt.tight_layout()
plt.savefig('/tmp/cv02_map.png', dpi=80)
plt.close()
print('Saved: /tmp/cv02_map.png')
